<a href="https://colab.research.google.com/github/kamalrawat77/agentic-iam-lab/blob/main/week08-agent-architecture/Nugget042_Adaptive_Workflows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nugget 042: LLM-Driven Workflow Decisions

In [34]:
!pip install -q google-genai
import json

In [48]:
from google import genai

def callGPT(prompt):
  client = genai.Client(api_key="APIKEY")
  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )

  return response

In [36]:
def cleanse_response(response):
  clean_response=clean_response = response.text
  clean_response = clean_response.replace("```json", "")
  clean_response = clean_response.replace("```", "")
  clean_response = clean_response.strip()

  return clean_response

In [37]:
def return_json(clean_response,objname):
  responseObj = json.loads(clean_response)
  if not objname:
    return responseObj
  resJsonObj  = responseObj[objname]
  return resJsonObj



Initial state:

In [53]:
current_state = "PLAN"

Create node functions.

In [39]:
def plan(state):

    print("Planning")
    print(state)
    return state


def retrieve(state):

    print("Retrieving")
    print(state)
    state["evidence"].append(
        "Dormant accounts increasing"
    )

    return state


def analyze(state):

    print("Analyzing")
    print(state)
    state["analysis"] = (
        "Likely JML backlog"
    )

    return state


def decide(state):

    print("Decision Complete")
    print(state)
    return state

Create node registry.

In [40]:
nodes = {
    "PLAN": plan,
    "RETRIEVE": retrieve,
    "ANALYZE": analyze,
    "DECIDE": decide
}

Shared state:

In [52]:
agent_state = {
    "question":
        "Why are dormant accounts increasing?",
    "evidence": [],
    "analysis": None
}

Create LLM planner function

In [49]:
def planner(agent_state,stateLabel):
  planner_prompt = f"""
  You are an investigation planner.

  Question:
  {agent_state["question"]}

  Current Investigation

  Evidence:
  {agent_state["evidence"]}

  Analysis:
  {agent_state["analysis"]}

  Rules:

  - If evidence is empty -> RETRIEVE
  - If evidence exists but analysis is None -> ANALYZE
  - If analysis exists -> FINISH

  Return JSON only.

  Example:

  {{"next_state":"RETRIEVE"}}
  """
  response=callGPT(planner_prompt)
  nextStateDecision=return_json(cleanse_response(response),None)
  print(nextStateDecision)
  nextState = nextStateDecision[stateLabel]
  return nextState

Execute graph.

In [54]:
while current_state != "FINISH":

    node = nodes[current_state]

    agent_state = node(agent_state)

    current_state = planner(agent_state,"next_state")

Planning
{'question': 'Why are dormant accounts increasing?', 'evidence': [], 'analysis': None}
{'next_state': 'RETRIEVE'}
Retrieving
{'question': 'Why are dormant accounts increasing?', 'evidence': [], 'analysis': None}
{'next_state': 'ANALYZE'}
Analyzing
{'question': 'Why are dormant accounts increasing?', 'evidence': ['Dormant accounts increasing'], 'analysis': None}
{'next_state': 'FINISH'}


In [55]:
print(agent_state)

{'question': 'Why are dormant accounts increasing?', 'evidence': ['Dormant accounts increasing'], 'analysis': 'Likely JML backlog'}
